# Bank Account Fraud Detection Pipeline

In [22]:
# import required libraries
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [23]:
# define required variable lists

# list of numeric features
numeric_features = [
    'name_email_similarity',
    'credit_risk_score',
    'proposed_credit_limit',
    'intended_balcon_amount',
    'prev_address_months_count',
    'date_of_birth_distinct_emails_4w',
    'current_address_months_count',
    'device_distinct_emails_8w',
    'income',
    'customer_age'
]

# list of catgorical features
categorical_features = [
    'prev_address_months_count_missing',
    'bank_months_count_missing',
    'intended_balcon_amount_missing',
    'foreign_request',
    'housing_status',
    'payment_type',
    'device_os',
    'keep_alive_session',
    'has_other_cards',
    'phone_home_valid',
    'source',
    'is_complete'
]

# list of columns with missing values.
missing_values_columns = [
    'prev_address_months_count',
    'current_address_months_count',
    'bank_months_count', 
    'device_distinct_emails_8w', 
    'intended_balcon_amount',
    'session_length_in_minutes'
]

missing_values_labels = [
    'prev_address_months_count_missing',
    'current_address_months_count_missing',
    'bank_months_count_missing', 
    'device_distinct_emails_8w_missing', 
    'intended_balcon_amount_missing',
    'session_length_in_minutes_missing'
]

# list of log transform candidate features
log_transform_candidates = [
    'proposed_credit_limit', 
    'intended_balcon_amount', 
    'current_address_months_count', 
    'prev_address_months_count', 
    'device_distinct_emails_8w'
]

# list of features to drop.
drop_features = [
    'intended_balcon_amount', 
    'prev_address_months_count', 
    'housing_status_BG', 
    'payment_type_AE',
    'payment_type_AC', 
    'intended_balcon_amount_missing_1', 
    'prev_address_months_count_missing_1', 
    'bank_months_count_missing_1'
]


In [24]:
# impute missing values
class NegativesToNan(BaseEstimator, TransformerMixin):
    """ This class converts negatives into nan. """
    def __init__(self, columns):
        """ Initialise the class with columns with missing values. """
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform negatives into nan."""
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
               X[col] = X[col].mask(X[col] < 0, np.nan)
        return X

In [25]:
# create missing value labels
class MissingLabels(BaseEstimator, TransformerMixin):
    """This class creates missing value labels."""
    def __init__(self, columns):
        """Initialise the class with columns with missing values. """
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform missing values by creating new label columns."""
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[f'{col}_missing'] = (X[col] < 0).astype(int)
        return X

In [26]:
# combine missing value indicators
class CombineMissingLabels(BaseEstimator, TransformerMixin):
    """This class combines missing value labels into one."""
    def __init__(self, columns):
        """Initialise the class with missing value label columns to be combined."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Combine missing labels into one."""
        X = X.copy()
        existing = [col for col in self.columns if col in X.columns]
        if existing:
            X['is_complete'] = X[existing].max(axis=1)
        return X

In [27]:
# log transform all numeric features
class LogTransformer(BaseEstimator, TransformerMixin):
    """This class log transform numeric data."""
    def __init__(self, columns):
        """Initialise the class with columns to be log transformed."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform numeric data."""
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[col] = np.log1p(X[col])
        return X

In [28]:
# drop unneeded features
class FeatureDropper(BaseEstimator, TransformerMixin):
    """This class drops unnecessary features."""
    def __init__(self, columns):
        """Initailise with features to drop."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Drop selected features."""
        X = X.copy()
        # drop only columns that exist
        existing = [col for col in self.columns if col in X.columns]
        
        return X.drop(columns=existing)

In [29]:
# convert array to dataframe
class ArrayToDataFrame(BaseEstimator, TransformerMixin):
    """This class converts arrays to dataframes."""
    def __init__(self, columns):
        """Initialise with columns to transform."""
        self.columns = columns

    def fit(self, X, y=None):
        """Do nothing in fit."""
        return self

    def transform(self, X):
        """Transform arrays into dataframes."""
        return pd.DataFrame(X, columns=self.columns)

In [30]:
class ColumnTransformerToDataFrame(BaseEstimator, TransformerMixin):
    """
    Convert ColumnTransformer output to DataFrame safely.
    """

    def __init__(self, column_transformer):
        self.column_transformer = column_transformer

    def fit(self, X, y=None):

        feature_names = []

        # iterate through fitted transformers
        for name, transformer, columns in self.column_transformer.transformers_:

            if name == 'remainder':
                continue

            # numeric pipeline → keep original names
            if name == 'numeric_features_transformation':
                feature_names.extend(columns)

            # categorical pipeline → get names from fitted encoder
            elif name == 'categorical_features_transformation':

                encoder = transformer.named_steps['encoder']

                cat_names = encoder.get_feature_names_out(columns)

                feature_names.extend(cat_names)

        self.feature_names_ = feature_names

        return self

    def transform(self, X):

        return pd.DataFrame(X, columns=self.feature_names_)

In [31]:
# numeric feature engineering pipeline
dataframe_pipeline = Pipeline([
    # create missing value labels 
    ('missing_values', MissingLabels(missing_values_columns)),
    # combine missing value indicator
    ('missing_indicator', CombineMissingLabels(missing_values_labels)),
    # convert negatives to nan
    ('negatives_to_nan', NegativesToNan(missing_values_columns)),
])

In [32]:
# numeric feature engineering pipeline
numeric_features_pipeline = Pipeline([
    # impute nan with median
    ('simple_imputer', SimpleImputer(strategy='median')),
    # to dataframe
    ('to_dataframe', ArrayToDataFrame(numeric_features)),
    # log transform numeric features
    ('log_transformer', LogTransformer(log_transform_candidates)),
    # scale all data
    ('scale', StandardScaler())
])

In [33]:
# categorical feature engineering pipeline
categorical_features_pipeline = Pipeline([

    ('encoder', OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    ))

])

In [34]:
# build preprocessor with ColumnTransformer
preprocessor = ColumnTransformer([
    # transform numeric features
    ('numeric_features_transformation', numeric_features_pipeline, numeric_features),
    # transform categorical features
    ('categorical_features_transformation', categorical_features_pipeline, categorical_features)
])

In [35]:
# build the logistic regression model pipeline
logistic_regression_pipeline = Pipeline([
    # dataframe 
    ('dataframe', dataframe_pipeline),
    # preprocessor
    ('preprocessor', preprocessor),
    # to dataframe
    ('preprocessor_to_dataframe', ColumnTransformerToDataFrame(preprocessor)),
    # drop features
    ('drop_features', FeatureDropper(drop_features)),
])

Test the pipeline

In [36]:
data = pd.read_csv(r"D:\LZMyDocs\Programme - Financial Crime Risk Analytics\Projects - Financial Crime Risk Analytics\Feedzai Bank Account Fraud Dataset\Base.csv")

In [37]:
X = data.drop('fraud_bool', axis=1)
y = data['fraud_bool']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [38]:
# FIT ONLY ON TRAIN DATA (CRITICAL FOR NO LEAKAGE)
X_train_processed = logistic_regression_pipeline.fit_transform(X_train)
# APPLY SAME TRANSFORM TO TEST DATA
X_test_processed = logistic_regression_pipeline.transform(X_test)

C:\Users\lizha\anaconda3\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
C:\Users\lizha\anaconda3\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [39]:
pd.set_option('display.max_columns', None) 

X_train_processed.head()

,name_email_similarity,credit_risk_score,proposed_credit_limit,date_of_birth_distinct_emails_4w,current_address_months_count,device_distinct_emails_8w,income,customer_age,foreign_request_1,housing_status_BB,housing_status_BC,housing_status_BD,housing_status_BE,housing_status_BF,payment_type_AB,payment_type_AD,device_os_macintosh,device_os_other,device_os_windows,device_os_x11,keep_alive_session_1,has_other_cards_1,phone_home_valid_1,source_TELEAPP,is_complete_1
0,-0.913479,-0.488158,1.791797,0.693812,0.236747,-0.069959,0.472338,1.355136,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,1.503397,0.186357,0.414335,-0.893964,-0.500646,-0.069959,-1.249763,3.017888,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,0.570158,-0.631672,-0.732169,-0.893964,-2.163898,-0.069959,-1.594184,0.523760,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,0.972195,1.076144,0.414335,-0.893964,0.749233,-0.069959,0.816758,-0.307615,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,1.728344,-0.244185,-0.732169,0.296868,-1.219435,-0.069959,1.161178,-1.138991,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0


In [40]:
X_train_processed.shape

(800000, 25)

In [41]:
X_train_processed.columns

Index(['name_email_similarity', 'credit_risk_score', 'proposed_credit_limit',
       'date_of_birth_distinct_emails_4w', 'current_address_months_count',
       'device_distinct_emails_8w', 'income', 'customer_age',
       'foreign_request_1', 'housing_status_BB', 'housing_status_BC',
       'housing_status_BD', 'housing_status_BE', 'housing_status_BF',
       'payment_type_AB', 'payment_type_AD', 'device_os_macintosh',
       'device_os_other', 'device_os_windows', 'device_os_x11',
       'keep_alive_session_1', 'has_other_cards_1', 'phone_home_valid_1',
       'source_TELEAPP', 'is_complete_1'],
      dtype='object')